[Course material - Sustain.Brussels - "Avdanced AI workflows with LLM" - 20/04/2026 - 22/04/2026](https://github.com/Yannael/gen-ai-sustain-brussels-2604).

# 2.6 — Benchmarking LLMs: Ground Truth vs. LLM-as-Judge


## Why Benchmark?

Before deploying an AI model in a real application — say, a chatbot answering sustainability questions for a city government — you need to know which model gives the best answers. But "best" is not obvious. Two questions immediately arise:

1. **Best at what?** Factual recall? Reasoning quality? Tone? Brevity?
2. **How do you measure it?** If there's a correct answer, you can check it. But what if the question requires judgment?

This tutorial introduces two complementary approaches:

| Approach | When to use | Metric |
|---|---|---|
| **MCQ with ground truth** | Factual questions with a known correct answer | Accuracy (correct / total) |
| **LLM-as-Judge** | Open-ended questions requiring reasoning or nuance | Score from a judge model (1–5) |

Both approaches are cheap to run, easy to automate, and directly comparable across models.

## Setup

Run the cell below to install dependencies (skip if already installed).

In [ ]:
# !pip install openai pandas

In [ ]:
import json
import pandas as pd
from openai import OpenAI

OPENROUTER_API_KEY = "..."

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODELS = [
    "mistralai/ministral-8b-2512",
    "meta-llama/llama-3.2-3b-instruct",
]
JUDGE_MODEL =   "qwen/qwen3-8b" #google/gemini-3-flash-preview"

print("Client ready. Models:", MODELS)

Client ready. Models: ['mistralai/ministral-8b-2512', 'meta-llama/llama-3.2-3b-instruct']


In [ ]:
MCQ_QUESTIONS = [
    {
        "id": "mcq1",
        "question": (
            "Approximately how much CO2-equivalent was emitted training GPT-3 "
            "(175B parameters) according to the widely-cited 2021 estimate "
            "by Patterson et al. (Google)?"
        ),
        "choices": {
            "A": "~5 tonnes CO2e",
            "B": "~500 tonnes CO2e",
            "C": "~5,000 tonnes CO2e",
            "D": "~50,000 tonnes CO2e",
        },
        "correct": "B",
    },
    {
        "id": "mcq2",
        "question": (
            "According to a 2023 UC Riverside study, approximately how much "
            "fresh water does ChatGPT consume per 20\u201350 user prompts?"
        ),
        "choices": {
            "A": "~10 millilitres",
            "B": "~100 millilitres",
            "C": "~500 millilitres",
            "D": "~5 litres",
        },
        "correct": "C",
    },
    {
        "id": "mcq3",
        "question": (
            "What phenomenon describes the observation that cheaper AI compute "
            "leads to *more* total AI usage, potentially offsetting efficiency gains?"
        ),
        "choices": {
            "A": "Dennard Scaling",
            "B": "Amdahl's Law",
            "C": "Jevons' Paradox",
            "D": "Moore's Law",
        },
        "correct": "C",
    },
]

OPEN_QUESTIONS = [
    {
        "id": "judge1",
        "short": "Carbon offsets",
        "question": (
            "Should AI companies be legally required to purchase carbon offsets "
            "for CO2 emitted during large model training? "
            "Argue your position with at least two concrete reasons."
        ),
    },
    {
        "id": "judge2",
        "short": "Jevons' Paradox",
        "question": (
            "Efficiency improvements mean training a model of given capability "
            "costs 100x less energy than 5 years ago. "
            "Does this necessarily reduce AI's total environmental impact? Explain."
        ),
    },
    {
        "id": "judge3",
        "short": "Small vs. large model",
        "question": (
            "A city government must choose between a locally-hosted 3B-parameter "
            "open-source model and a large proprietary API (70B+) for processing "
            "citizen requests. What are the environmental trade-offs?"
        ),
    },
]

JUDGE_SYSTEM_PROMPT = """\
You are an expert evaluator assessing responses about AI and environmental sustainability.
Evaluate the response on three criteria and respond ONLY in valid JSON — no other text.

Criteria:
- accuracy (1-5): Does the response reflect factually correct information about AI energy consumption, environmental impact, or relevant policy?
- nuance (1-5): Does the response acknowledge trade-offs, competing considerations, or limitations of its own argument?
- actionability (1-5): Does the response include concrete, specific recommendations or mechanisms (not just abstract principles)?

Respond with exactly this structure:
{
  \"accuracy\": <int 1-5>,
  \"accuracy_reason\": \"<one sentence>\",
  \"nuance\": <int 1-5>,
  \"nuance_reason\": \"<one sentence>\",
  \"actionability\": <int 1-5>,
  \"actionability_reason\": \"<one sentence>\"
}
"""

print(f"Loaded {len(MCQ_QUESTIONS)} MCQ questions and {len(OPEN_QUESTIONS)} open questions.")

Loaded 3 MCQ questions and 3 open questions.


---

## Part 1 — MCQ Benchmark (Ground Truth)

### How it works

We send each question to each model with a strict system prompt: *"Reply with exactly one letter: A, B, C, or D."* This forces the model to commit to a single answer, making evaluation trivial — no regex, no parsing ambiguity.

We use `temperature=0` to make results deterministic. We set `max_tokens=5` to prevent the model from rambling.

The two functions below follow a clean separation:
- `ask_mcq()` handles one question for one model
- `run_mcq_benchmark()` loops over all combinations and returns a DataFrame

In [ ]:
def ask_mcq(question_text: str, choices: dict, model: str) -> str:
    choices_text = "\n".join(f"{k}) {v}" for k, v in choices.items())
    user_message = f"{question_text}\n\n{choices_text}"

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Reply with exactly one letter: A, B, C, or D. No other text.",
            },
            {"role": "user", "content": user_message},
        ],
        max_tokens=5,
        temperature=0,
    )
    answer = response.choices[0].message.content.strip().upper()
    return answer[0] if answer else "?"


def run_mcq_benchmark(mcq_questions: list, models: list) -> pd.DataFrame:
    rows = []
    for model in models:
        for q in mcq_questions:
            print(f"  {model.split('/')[-1]} / {q['id']}...", end=" ")
            model_answer = ask_mcq(q["question"], q["choices"], model)

            is_correct = model_answer == q["correct"]
            print(f"{model_answer} ({'✓' if is_correct else '✗'})")
            rows.append({
                "model": model.split("/")[-1],
                "question_id": q["id"],
                "model_answer": model_answer,
                "correct_answer": q["correct"],
                "correct": is_correct,
            })
    return pd.DataFrame(rows)

In [ ]:
print("Running MCQ benchmark...")
df_mcq = run_mcq_benchmark(MCQ_QUESTIONS, MODELS)

print("\n--- All answers ---")
display(df_mcq)

print("\n--- Accuracy per model ---")
accuracy = df_mcq.groupby("model")["correct"].mean().rename("accuracy").reset_index()
accuracy["accuracy"] = accuracy["accuracy"].apply(lambda x: f"{x:.0%}")
display(accuracy)

Running MCQ benchmark...
  ministral-8b-2512 / mcq1... B (✓)
  ministral-8b-2512 / mcq2... B (✗)
  ministral-8b-2512 / mcq3... C (✓)
  llama-3.2-3b-instruct / mcq1... A (✗)
  llama-3.2-3b-instruct / mcq2... I (✗)
  llama-3.2-3b-instruct / mcq3... T (✗)

--- All answers ---


,model,question_id,model_answer,correct_answer,correct
0,ministral-8b-2512,mcq1,B,B,True
1,ministral-8b-2512,mcq2,B,C,False
2,ministral-8b-2512,mcq3,C,C,True
3,llama-3.2-3b-instruct,mcq1,A,B,False
4,llama-3.2-3b-instruct,mcq2,I,C,False
5,llama-3.2-3b-instruct,mcq3,T,C,False



--- Accuracy per model ---


,model,accuracy
0,llama-3.2-3b-instruct,0%
1,ministral-8b-2512,67%


> ### 💬 Reflection 1
>
> Look at the accuracy scores above. A model that scores 2/3 is very different from one that scores 3/3 — but how much can you trust a 3-question benchmark?
>
> **Discuss with your neighbour:** Name at least two failure modes of a small MCQ benchmark.
> Think about: how the questions were written, how models were trained, and what "knowing the answer" really means for an LLM.
>
> *(Hints if you're stuck: data contamination, sampling variance, question wording bias)*

---

## Part 2 — LLM-as-Judge Benchmark

### The problem with MCQ

MCQ works well for factual recall. But the most important questions in AI & sustainability are not factual:

- *Should* companies be required to offset emissions?
- *How should* governments balance innovation and environmental cost?

These require reasoning, trade-off analysis, and nuance — things a letter grade cannot capture.

### The LLM-as-Judge approach

Instead of checking against a ground truth, we ask a third model (the "judge") to score each response on explicit criteria. This is the same principle as human expert evaluation, but automated and cheap.

The key to a good judge is a **precise scoring rubric**. Our rubric uses three dimensions:

| Dimension | Measures |
|---|---|
| **Accuracy** | Are the facts about AI energy/environment correct? |
| **Nuance** | Does it acknowledge trade-offs and limitations? |
| **Actionability** | Does it recommend concrete steps, not just principles? |

We ask the judge to return structured JSON so parsing is deterministic.

In [ ]:
def ask_open(question_text: str, model: str) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant with expertise in AI and environmental sustainability.",
            },
            {"role": "user", "content": question_text},
        ],
        max_tokens=300,
        temperature=0.7,
    )
    return response.choices[0].message.content.strip()


def judge_response(question_text: str, response_text: str, judge_model: str) -> dict:
    user_message = f"Question: {question_text}\n\nResponse to evaluate:\n{response_text}"

    judgment = client.chat.completions.create(
        model=judge_model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        max_tokens=400,
        temperature=0,
    )
    raw = judgment.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {
            "accuracy": None, "accuracy_reason": "parse error",
            "nuance": None, "nuance_reason": "parse error",
            "actionability": None, "actionability_reason": "parse error",
        }


def run_judge_benchmark(open_questions: list, models: list, judge_model: str) -> pd.DataFrame:
    rows = []
    for model in models:
        for q in open_questions:
            print(f"  {model.split('/')[-1]} / {q['id']}...", end=" ")
            response_text = ask_open(q["question"], model)
            print(response_text)
            scores = judge_response(q["question"], response_text, judge_model)
            accuracy = scores.get("accuracy")
            nuance = scores.get("nuance")
            actionability = scores.get("actionability")
            total = (
                (accuracy or 0) + (nuance or 0) + (actionability or 0)
                if all(v is not None for v in [accuracy, nuance, actionability])
                else None
            )
            print(f"acc={accuracy}, nuance={nuance}, action={actionability}")
            rows.append({
                "model": model.split("/")[-1],
                "question_id": q["id"],
                "question_short": q["short"],
                "response_preview": response_text[:80] + "...",
                "accuracy": accuracy,
                "nuance": nuance,
                "actionability": actionability,
                "total_score": total,
            })
    return pd.DataFrame(rows)

In [ ]:
print("Running LLM-as-Judge benchmark...")
df_judge = run_judge_benchmark(OPEN_QUESTIONS, MODELS, JUDGE_MODEL)

print("\n--- Scores per response ---")
display(df_judge[["model", "question_short", "accuracy", "nuance", "actionability", "total_score"]])

print("\n--- Average scores per model ---")
display(
    df_judge.groupby("model")[["accuracy", "nuance", "actionability", "total_score"]]
    .mean()
    .round(2)
)

Running LLM-as-Judge benchmark...
  ministral-8b-2512 / judge1... Yes, AI companies should be legally required to purchase carbon offsets for the CO₂ emissions generated during large-scale model training. Here are two concrete reasons to support this position:

### 1. **Corporate Accountability and Environmental Justice**
   - **Mandatory offsets force transparency and responsibility**: AI training—particularly for large language models (LLMs) and foundation models—is energy-intensive, often relying on fossil fuel-powered data centers. Requiring offsets would compel companies to **publicly disclose their emissions** and demonstrate efforts to mitigate harm. This aligns with principles of **corporate accountability**, ensuring that companies cannot externalize the environmental costs of their operations (e.g., carbon pollution) onto marginalized communities, who disproportionately bear the brunt of climate impacts.
   - **Prevents "greenwashing"**: Without regulation, companies may clai

,model,question_short,accuracy,nuance,actionability,total_score
0,ministral-8b-2512,Carbon offsets,4,3,3,10
1,ministral-8b-2512,Jevons' Paradox,5,4,2,11
2,ministral-8b-2512,Small vs. large model,4,3,2,9
3,qwen3-8b,Carbon offsets,4,3,3,10
4,qwen3-8b,Jevons' Paradox,4,4,2,10
5,qwen3-8b,Small vs. large model,4,3,2,9



--- Average scores per model ---


,accuracy,nuance,actionability,total_score
model,,,,
ministral-8b-2512,4.33,3.33,2.33,10.00
qwen3-8b,4.00,3.33,2.33,9.67


> ### 💬 Reflection 2
>
> The judge above is itself an LLM — `qwen/qwen3-8b`. This creates a potential problem.
>
> **Discuss:** What systematic biases might a judge LLM have when scoring another model's response?
> Does it matter if the judge and the candidate come from the same model family (e.g., both Qwen)?
> What would a more rigorous evaluation design look like?
>
> *(Hints: verbosity bias, self-enhancement bias, the judge's own priors on sustainability policy)*

---

## Part 3 — Summary and Coding Exercise

### Combined results

The cell below merges both benchmarks into a single comparison table — a minimal benchmark report.

In [ ]:
mcq_summary = (
    df_mcq.groupby("model")["correct"]
    .mean()
    .rename("mcq_accuracy")
    .reset_index()
)
judge_summary = (
    df_judge.groupby("model")["total_score"]
    .mean()
    .rename("avg_judge_score")
    .reset_index()
)
summary = mcq_summary.merge(judge_summary, on="model")
summary["mcq_accuracy"] = summary["mcq_accuracy"].apply(lambda x: f"{x:.0%}")
summary["avg_judge_score"] = summary["avg_judge_score"].round(1)

print("=== BENCHMARK SUMMARY ===")
display(summary)

=== BENCHMARK SUMMARY ===


,model,mcq_accuracy,avg_judge_score
0,ministral-8b-2512,67%,11.3
1,qwen3-8b,67%,10.0


> ### ✏️ Coding Exercise
>
> **Add a third model to the benchmark.**
>
> 1. Go to [openrouter.ai/models](https://openrouter.ai/models) and pick any model — for example `meta-llama/llama-3.2-3b-instruct` or `google/gemma-3-4b-it`.
> 2. Add it to the `MODELS` list in the cell below.
> 3. Re-run `run_mcq_benchmark()` and `run_judge_benchmark()` and the summary cell.
> 4. Does the ranking change? Is the new model better on MCQ, on judge scores, or both?
>
> **Notice:** You did not change any function code — only the data list. This is the payoff of keeping data and logic separate.

In [ ]:
# TODO: add your model here
MODELS_EXTENDED = [
    "mistral/mistral-small-3.2",
    "qwen/qwen3-8b",
    # "meta-llama/llama-3.2-3b-instruct",  # <-- uncomment and change this
]

df_mcq_ext = run_mcq_benchmark(MCQ_QUESTIONS, MODELS_EXTENDED)
df_judge_ext = run_judge_benchmark(OPEN_QUESTIONS, MODELS_EXTENDED, JUDGE_MODEL)

mcq_sum2 = df_mcq_ext.groupby("model")["correct"].mean().rename("mcq_accuracy").reset_index()
judge_sum2 = df_judge_ext.groupby("model")["total_score"].mean().rename("avg_judge_score").reset_index()
summary2 = mcq_sum2.merge(judge_sum2, on="model")
summary2["mcq_accuracy"] = summary2["mcq_accuracy"].apply(lambda x: f"{x:.0%}")
summary2["avg_judge_score"] = summary2["avg_judge_score"].round(1)

print("=== EXTENDED BENCHMARK SUMMARY ===")
display(summary2)

---

## Key Takeaways

| | MCQ (ground truth) | LLM-as-Judge |
|---|---|---|
| **Measures** | Factual recall | Reasoning quality |
| **Requires** | Known correct answers | A judge model + rubric |
| **Strengths** | Objective, reproducible | Flexible, captures nuance |
| **Weaknesses** | Can't evaluate open-ended reasoning | Judge has its own biases |
| **Best for** | Knowledge benchmarks | Policy / ethics questions |
